In [22]:
import os
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# DATASET PATH
# ============================================================

DATASET_PATH = "datasets/stop"


# ============================================================
# IMAGE TRANSFORM
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


# ============================================================
# LOAD DATASET
# ============================================================

dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=transform
)


# ============================================================
# DATASET INFORMATION
# ============================================================

print()
print("Classes:", dataset.classes)
print("Class mapping:", dataset.class_to_idx)
print("Total images:", len(dataset))

print()

for class_name in dataset.classes:

    folder = os.path.join(
        DATASET_PATH,
        class_name
    )

    count = len([
        f for f in os.listdir(folder)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ])

    print(
        class_name,
        ":",
        count
    )

Device: cuda

Classes: ['close', 'far', 'none']
Class mapping: {'close': 0, 'far': 1, 'none': 2}
Total images: 1459

close : 149
far : 114
none : 1142


In [23]:
from collections import Counter

# ============================================================
# REAL COUNTS USED BY IMAGEFOLDER
# ============================================================

target_counts = Counter(dataset.targets)

print("Actual images used by ImageFolder:")
print()

for class_name, class_index in dataset.class_to_idx.items():

    print(
        class_name,
        ":",
        target_counts[class_index]
    )

print()
print("TOTAL:", sum(target_counts.values()))

Actual images used by ImageFolder:

close : 165
far : 127
none : 1167

TOTAL: 1459


In [24]:
import random
from torch.utils.data import Subset, DataLoader

# ============================================================
# STRATIFIED TRAIN / VALIDATION SPLIT
# ============================================================

random.seed(42)

# אוספים את האינדקסים של כל מחלקה
indices_by_class = {
    0: [],   # CLOSE
    1: [],   # FAR
    2: []    # NONE
}

for index, target in enumerate(dataset.targets):
    indices_by_class[target].append(index)


train_indices = []
val_indices = []

VAL_RATIO = 0.20


for class_index, indices in indices_by_class.items():

    random.shuffle(indices)

    val_count = int(len(indices) * VAL_RATIO)

    val_indices.extend(
        indices[:val_count]
    )

    train_indices.extend(
        indices[val_count:]
    )


# מערבבים את האינדקסים
random.shuffle(train_indices)
random.shuffle(val_indices)


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = Subset(
    dataset,
    train_indices
)

val_dataset = Subset(
    dataset,
    val_indices
)


# ============================================================
# DATA LOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)


# ============================================================
# CHECK SPLIT
# ============================================================

train_counts = {
    0: 0,
    1: 0,
    2: 0
}

val_counts = {
    0: 0,
    1: 0,
    2: 0
}


for index in train_indices:
    target = dataset.targets[index]
    train_counts[target] += 1


for index in val_indices:
    target = dataset.targets[index]
    val_counts[target] += 1


print("TRAIN:")
print("CLOSE:", train_counts[0])
print("FAR  :", train_counts[1])
print("NONE :", train_counts[2])
print("TOTAL:", len(train_dataset))

print()

print("VALIDATION:")
print("CLOSE:", val_counts[0])
print("FAR  :", val_counts[1])
print("NONE :", val_counts[2])
print("TOTAL:", len(val_dataset))

TRAIN:
CLOSE: 132
FAR  : 102
NONE : 934
TOTAL: 1168

VALIDATION:
CLOSE: 33
FAR  : 25
NONE : 233
TOTAL: 291


In [4]:
# ============================================================
# NEW STOP CNN - TRAIN FROM SCRATCH
# ============================================================

class StopCNN(nn.Module):
    def __init__(self):
        super(StopCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ============================================================
# CREATE BRAND-NEW MODEL
# ============================================================

model = StopCNN().to(device)

print(model)

print()
print("NEW model created from scratch!")

StopCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(2, 2))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(5, 5), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2))
    (5): ReLU()
    (6): AdaptiveAvgPool2d(output_size=(4, 4))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1024, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=3, bias=True)
  )
)

NEW model created from scratch!


In [25]:
# ============================================================
# STOP CNN
# ============================================================

class StopCNN(nn.Module):
    def __init__(self):
        super(StopCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ============================================================
# LOAD PREVIOUS BEST MODEL
# ============================================================

model = StopCNN().to(device)

model.load_state_dict(
    torch.load(
        "models/best_traffic_stop_model.pth",
        map_location=device
    )
)

model.eval()

print("Previous best traffic stop model loaded!")
print("Ready for fine-tuning.")

Previous best traffic stop model loaded!
Ready for fine-tuning.


In [26]:
# ============================================================
# FINE-TUNING SETTINGS
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

NUM_EPOCHS = 15

BEST_MODEL_PATH = "models/best_traffic_stop_model_v2.pth"

best_val_loss = float("inf")
best_epoch = 0


# ============================================================
# FINE-TUNING
# ============================================================

for epoch in range(NUM_EPOCHS):

    # ---------------- TRAIN ----------------

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (
            predicted == labels
        ).sum().item()


    train_loss /= train_total

    train_acc = (
        100.0 *
        train_correct /
        train_total
    )


    # ---------------- VALIDATION ----------------

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()


    val_loss /= val_total

    val_acc = (
        100.0 *
        val_correct /
        val_total
    )


    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )


    # ---------------- SAVE BEST ----------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print(
            "   -> BEST FINE-TUNED MODEL SAVED"
        )


# ============================================================
# RESULT
# ============================================================

print()
print("Fine-tuning finished!")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Saved as:", BEST_MODEL_PATH)

Epoch 01/15 | Train Loss: 0.3310 | Train Acc: 89.81% | Val Loss: 0.3173 | Val Acc: 86.60%
   -> BEST FINE-TUNED MODEL SAVED
Epoch 02/15 | Train Loss: 0.3239 | Train Acc: 89.04% | Val Loss: 0.3124 | Val Acc: 87.29%
   -> BEST FINE-TUNED MODEL SAVED
Epoch 03/15 | Train Loss: 0.3173 | Train Acc: 89.73% | Val Loss: 0.2965 | Val Acc: 88.32%
   -> BEST FINE-TUNED MODEL SAVED
Epoch 04/15 | Train Loss: 0.2976 | Train Acc: 89.38% | Val Loss: 0.2821 | Val Acc: 87.63%
   -> BEST FINE-TUNED MODEL SAVED
Epoch 05/15 | Train Loss: 0.2765 | Train Acc: 91.27% | Val Loss: 0.2648 | Val Acc: 90.03%
   -> BEST FINE-TUNED MODEL SAVED
Epoch 06/15 | Train Loss: 0.2604 | Train Acc: 91.44% | Val Loss: 0.2491 | Val Acc: 90.72%
   -> BEST FINE-TUNED MODEL SAVED
Epoch 07/15 | Train Loss: 0.2493 | Train Acc: 91.87% | Val Loss: 0.2433 | Val Acc: 91.07%
   -> BEST FINE-TUNED MODEL SAVED
Epoch 08/15 | Train Loss: 0.2518 | Train Acc: 91.87% | Val Loss: 0.2392 | Val Acc: 91.07%
   -> BEST FINE-TUNED MODEL SAVED
Epoch 09

In [5]:
# ============================================================
# LOSS + OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

NUM_EPOCHS = 20

BEST_MODEL_PATH = "models/best_traffic_stop_model.pth"

best_val_loss = float("inf")
best_epoch = 0


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)


        optimizer.zero_grad()


        outputs = model(images)


        loss = criterion(
            outputs,
            labels
        )


        loss.backward()

        optimizer.step()


        train_loss += (
            loss.item() * images.size(0)
        )


        _, predicted = torch.max(
            outputs,
            1
        )


        train_total += labels.size(0)

        train_correct += (
            predicted == labels
        ).sum().item()


    train_loss /= train_total

    train_acc = (
        100.0
        * train_correct
        / train_total
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)


            outputs = model(images)


            loss = criterion(
                outputs,
                labels
            )


            val_loss += (
                loss.item()
                * images.size(0)
            )


            _, predicted = torch.max(
                outputs,
                1
            )


            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()


    val_loss /= val_total

    val_acc = (
        100.0
        * val_correct
        / val_total
    )


    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print(
            "   -> BEST MODEL SAVED"
        )


# ============================================================
# FINISHED
# ============================================================

print()
print("Training finished!")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Saved as:", BEST_MODEL_PATH)

Epoch 01/20 | Train Loss: 0.5731 | Train Acc: 83.58% | Val Loss: 0.4962 | Val Acc: 85.87%
   -> BEST MODEL SAVED
Epoch 02/20 | Train Loss: 0.4980 | Train Acc: 85.61% | Val Loss: 0.4538 | Val Acc: 85.87%
   -> BEST MODEL SAVED
Epoch 03/20 | Train Loss: 0.4446 | Train Acc: 85.98% | Val Loss: 0.3930 | Val Acc: 88.85%
   -> BEST MODEL SAVED
Epoch 04/20 | Train Loss: 0.4177 | Train Acc: 86.99% | Val Loss: 0.4050 | Val Acc: 85.87%
Epoch 05/20 | Train Loss: 0.3863 | Train Acc: 87.55% | Val Loss: 0.3606 | Val Acc: 88.85%
   -> BEST MODEL SAVED
Epoch 06/20 | Train Loss: 0.3835 | Train Acc: 88.10% | Val Loss: 0.3811 | Val Acc: 88.85%
Epoch 07/20 | Train Loss: 0.3771 | Train Acc: 88.56% | Val Loss: 0.3612 | Val Acc: 87.73%
Epoch 08/20 | Train Loss: 0.3647 | Train Acc: 87.92% | Val Loss: 0.3234 | Val Acc: 89.59%
   -> BEST MODEL SAVED
Epoch 09/20 | Train Loss: 0.3371 | Train Acc: 88.84% | Val Loss: 0.3067 | Val Acc: 89.96%
   -> BEST MODEL SAVED
Epoch 10/20 | Train Loss: 0.3140 | Train Acc: 89.21%

In [27]:
# ============================================================
# LOAD BEST FINE-TUNED MODEL V2
# ============================================================

model.load_state_dict(
    torch.load(
        "models/best_traffic_stop_model_v2.pth",
        map_location=device
    )
)

model.eval()

print("Best traffic stop V2 model loaded!")

Best traffic stop V2 model loaded!


In [6]:
# ============================================================
# LOAD BEST TRAFFIC STOP MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        "models/best_traffic_stop_model.pth",
        map_location=device
    )
)

model.eval()

print("Best traffic stop model loaded!")

Best traffic stop model loaded!


In [28]:
# ============================================================
# PER-CLASS VALIDATION TEST
# ============================================================

class_names = [
    "CLOSE",   # 0
    "FAR",     # 1
    "NONE"     # 2
]

correct_per_class = [0, 0, 0]
total_per_class = [0, 0, 0]


with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(
            outputs,
            1
        )

        for label, prediction in zip(labels, predicted):

            label_index = label.item()
            prediction_index = prediction.item()

            total_per_class[label_index] += 1

            if prediction_index == label_index:
                correct_per_class[label_index] += 1


print()
print("PER-CLASS VALIDATION RESULTS")
print("============================")

for i, class_name in enumerate(class_names):

    correct = correct_per_class[i]
    total = total_per_class[i]

    accuracy = (
        100.0 * correct / total
        if total > 0
        else 0.0
    )

    print(
        f"{class_name}: "
        f"{correct}/{total} "
        f"= {accuracy:.2f}%"
    )


PER-CLASS VALIDATION RESULTS
CLOSE: 19/33 = 57.58%
FAR: 18/25 = 72.00%
NONE: 232/233 = 99.57%


In [29]:
# ============================================================
# CONFUSION CHECK
# ============================================================

confusion = [
    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0]
]

class_names = [
    "CLOSE",
    "FAR",
    "NONE"
]

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(
            outputs,
            1
        )

        for true_label, pred_label in zip(
            labels,
            predicted
        ):

            true_index = true_label.item()
            pred_index = pred_label.item()

            confusion[true_index][pred_index] += 1


print()
print("CONFUSION MATRIX")
print("================")
print()
print("          Pred CLOSE   Pred FAR   Pred NONE")

for i, class_name in enumerate(class_names):

    print(
        f"True {class_name:<5} "
        f"{confusion[i][0]:>10} "
        f"{confusion[i][1]:>10} "
        f"{confusion[i][2]:>11}"
    )


CONFUSION MATRIX

          Pred CLOSE   Pred FAR   Pred NONE
True CLOSE         19          3          11
True FAR            3         18           4
True NONE           1          0         232


In [33]:
from jetbot import Camera

camera = Camera.instance(
    width=224,
    height=224
)

print("Camera ready!")

Camera ready!


In [34]:
import os
import time
import threading
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

try:
    robot.stop()
except:
    pass

class StopCNN(nn.Module):
    def __init__(self):
        super(StopCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = StopCNN().to(device)

model.load_state_dict(
    torch.load(
        "models/best_traffic_stop_model_v2.pth",
        map_location=device
    )
)

model.eval()

class_names = [
    "CLOSE",
    "FAR",
    "NONE"
]

print("NEW STOP V2 model loaded!")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

none_dir = "datasets/stop/none"
far_dir = "datasets/stop/far"
close_dir = "datasets/stop/close"

os.makedirs(none_dir, exist_ok=True)
os.makedirs(far_dir, exist_ok=True)
os.makedirs(close_dir, exist_ok=True)

CLASS_FOLDERS = {
    "NONE": none_dir,
    "FAR": far_dir,
    "CLOSE": close_dir
}

robot = Robot()

LEFT_GAIN = 1.035
RIGHT_GAIN = 1.00
STARTUP_SPEED = 0.18
STARTUP_TIME = 0.15

drive_direction = 0

camera_view = widgets.Image(
    format="jpeg",
    width=300,
    height=300
)

prediction_label = widgets.Label(
    value="Prediction: ---"
)

confidence_label = widgets.Label(
    value="Confidence: ---"
)

speed_slider = widgets.FloatSlider(
    value=0.09,
    min=0.05,
    max=0.30,
    step=0.01,
    description="Speed:"
)

steering = widgets.FloatSlider(
    value=0.0,
    min=-1.0,
    max=1.0,
    step=0.01,
    description="Steering:"
)

start_button = widgets.Button(
    description="START",
    button_style="success"
)

stop_button = widgets.Button(
    description="STOP",
    button_style="danger"
)

back_button = widgets.Button(
    description="BACK"
)

record_armed = False
recording = False
record_class = None
RECORD_INTERVAL = 0.20
current_prediction = None
record_start_prediction = None

def get_images(folder):
    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

none_count = widgets.IntText(
    description="NONE:",
    disabled=True
)

far_count = widgets.IntText(
    description="FAR:",
    disabled=True
)

close_count = widgets.IntText(
    description="CLOSE:",
    disabled=True
)

def update_counts():
    none_count.value = len(get_images(none_dir))
    far_count.value = len(get_images(far_dir))
    close_count.value = len(get_images(close_dir))

update_counts()

def save_live_image(class_name):
    folder = CLASS_FOLDERS[class_name]
    frame = camera.value

    if frame is None:
        print("No camera frame!")
        return

    filename = str(int(time.time() * 1000)) + ".jpg"
    path = os.path.join(folder, filename)

    with open(path, "wb") as f:
        f.write(bgr8_to_jpeg(frame))

    update_counts()
    print(f"Saved as {class_name}:", filename)

save_none_button = widgets.Button(description="SAVE NONE")
save_far_button = widgets.Button(description="SAVE FAR")
save_close_button = widgets.Button(description="SAVE CLOSE")

def save_none_live(b):
    save_live_image("NONE")

def save_far_live(b):
    save_live_image("FAR")

def save_close_live(b):
    save_live_image("CLOSE")

save_none_button.on_click(save_none_live)
save_far_button.on_click(save_far_live)
save_close_button.on_click(save_close_live)

record_none_button = widgets.Button(description="RECORD NONE")
record_far_button = widgets.Button(description="RECORD FAR")
record_close_button = widgets.Button(description="RECORD CLOSE")
stop_record_button = widgets.Button(description="STOP RECORD")

record_status = widgets.Label(
    value="Record: OFF"
)

def arm_record(class_name):
    global record_armed, recording, record_class, record_start_prediction

    recording = False
    record_armed = True
    record_class = class_name
    record_start_prediction = None

    record_status.value = f"ARMED: {class_name} | Press START"

    print(
        "RECORD ARMED:",
        class_name,
        "- waiting for START"
    )

def record_none(b):
    arm_record("NONE")

def record_far(b):
    arm_record("FAR")

def record_close(b):
    arm_record("CLOSE")

record_none_button.on_click(record_none)
record_far_button.on_click(record_far)
record_close_button.on_click(record_close)

def record_loop():
    global recording

    while recording:
        if drive_direction != 1:
            time.sleep(0.02)
            continue

        if record_class is None:
            time.sleep(0.02)
            continue

        frame = camera.value

        if frame is not None:
            folder = CLASS_FOLDERS[record_class]
            filename = str(int(time.time() * 1000)) + ".jpg"
            path = os.path.join(folder, filename)

            with open(path, "wb") as f:
                f.write(bgr8_to_jpeg(frame))

            update_counts()

        time.sleep(RECORD_INTERVAL)

def start_actual_recording():
    global recording, record_start_prediction

    if not record_armed:
        return

    if record_class is None:
        return

    if current_prediction is None:
        print("Waiting for prediction...")
        return

    if recording:
        return

    record_start_prediction = current_prediction
    recording = True

    record_status.value = (
        f"RECORDING: {record_class} | "
        f"Start Prediction: {record_start_prediction}"
    )

    threading.Thread(
        target=record_loop,
        daemon=True
    ).start()

    print(
        "RECORD STARTED:",
        record_class,
        "| Prediction:",
        record_start_prediction
    )

def stop_actual_recording(
    keep_armed=True,
    message="RECORD STOPPED"
):
    global recording, record_armed, record_class, record_start_prediction

    recording = False
    record_start_prediction = None

    if keep_armed and record_class is not None:
        record_status.value = (
            f"ARMED: {record_class} | Press START"
        )
    else:
        record_armed = False
        record_class = None
        record_status.value = "Record: OFF"

    print(message)

def stop_record(b=None):
    stop_actual_recording(
        keep_armed=False,
        message="RECORD DISARMED"
    )

stop_record_button.on_click(stop_record)

def update_motors(change=None):
    global drive_direction

    if drive_direction == 0:
        robot.stop()
        return

    speed = speed_slider.value
    s = steering.value

    steering_power = s * 0.10

    left_speed = (speed + steering_power) * LEFT_GAIN
    right_speed = (speed - steering_power) * RIGHT_GAIN

    left_speed *= drive_direction
    right_speed *= drive_direction

    left_speed = max(-1.0, min(1.0, left_speed))
    right_speed = max(-1.0, min(1.0, right_speed))

    robot.left_motor.value = left_speed
    robot.right_motor.value = right_speed

def start_drive(b):
    global drive_direction

    drive_direction = 1

    robot.left_motor.value = STARTUP_SPEED * LEFT_GAIN
    robot.right_motor.value = STARTUP_SPEED * RIGHT_GAIN

    time.sleep(STARTUP_TIME)

    update_motors()
    start_actual_recording()

    print("FORWARD")

def stop_drive(b):
    global drive_direction

    drive_direction = 0
    robot.stop()

    if recording:
        stop_actual_recording(
            keep_armed=True,
            message="RECORD STOPPED WITH ROBOT"
        )

    print("STOPPED")

def back_drive(b):
    global drive_direction

    if recording:
        stop_actual_recording(
            keep_armed=True,
            message="RECORD STOPPED - BACKWARD"
        )

    drive_direction = -1
    update_motors()

    print("BACKWARD")

start_button.on_click(start_drive)
stop_button.on_click(stop_drive)
back_button.on_click(back_drive)

speed_slider.observe(
    update_motors,
    names="value"
)

steering.observe(
    update_motors,
    names="value"
)

def check_prediction_change(predicted_class):
    global recording, record_start_prediction, drive_direction

    if not recording:
        return

    if record_start_prediction is None:
        return

    if predicted_class == record_start_prediction:
        record_status.value = (
            f"RECORDING: {record_class} | "
            f"Prediction: {predicted_class}"
        )
        return

    old_prediction = record_start_prediction
    new_prediction = predicted_class

    stop_actual_recording(
        keep_armed=True,
        message=(
            f"AUTO STOP RECORD: "
            f"{old_prediction} -> {new_prediction}"
        )
    )

    drive_direction = 0
    robot.stop()

    record_status.value = (
        f"AUTO STOP: "
        f"{old_prediction} -> "
        f"{new_prediction} | "
        f"ROBOT STOPPED"
    )

    print(
        "PREDICTION CHANGED:",
        old_prediction,
        "->",
        new_prediction,
        "| RECORD STOPPED | ROBOT STOPPED"
    )

delete_class = widgets.Dropdown(
    options=[
        "NONE",
        "FAR",
        "CLOSE"
    ],
    value="NONE",
    description="Class:"
)

delete_last_button = widgets.Button(
    description="DELETE LAST"
)

def delete_last_image(b):
    if recording:
        print("STOP RECORD before deleting.")
        return

    selected_class = delete_class.value
    folder = CLASS_FOLDERS[selected_class]
    images = get_images(folder)

    if len(images) == 0:
        print(selected_class, "folder is empty.")
        return

    filename = images[-1]

    os.remove(
        os.path.join(
            folder,
            filename
        )
    )

    update_counts()

    print(
        "Deleted:",
        selected_class,
        filename
    )

delete_last_button.on_click(delete_last_image)

from_box = widgets.IntText(
    value=1,
    description="From:"
)

to_box = widgets.IntText(
    value=1,
    description="To:"
)

delete_range_button = widgets.Button(
    description="DELETE RANGE"
)

def delete_range(b):
    if recording:
        print("STOP RECORD before deleting.")
        return

    selected_class = delete_class.value
    folder = CLASS_FOLDERS[selected_class]
    images = get_images(folder)

    start = from_box.value
    end = to_box.value

    if start < 1:
        print("From must be >= 1")
        return

    if end < start:
        print("To must be >= From")
        return

    if start > len(images):
        print("Start image does not exist.")
        return

    end = min(end, len(images))

    images_to_delete = images[start - 1:end]

    for filename in images_to_delete:
        os.remove(
            os.path.join(
                folder,
                filename
            )
        )

    update_counts()

    print(
        "Deleted",
        len(images_to_delete),
        "images from",
        selected_class,
        "range",
        start,
        "-",
        end
    )

delete_range_button.on_click(delete_range)

image_input = widgets.Text(
    description="Image:",
    placeholder="number or filename"
)

delete_image_button = widgets.Button(
    description="DELETE IMAGE"
)

def delete_specific_image(b):
    if recording:
        print("STOP RECORD before deleting.")
        return

    selected_class = delete_class.value
    folder = CLASS_FOLDERS[selected_class]
    images = get_images(folder)

    value = image_input.value.strip()

    if value == "":
        print("Enter image number or filename.")
        return

    if value.isdigit():
        image_number = int(value)

        if image_number < 1 or image_number > len(images):
            print("Image number does not exist.")
            return

        filename = images[image_number - 1]

    else:
        filename = value

        if filename not in images:
            print("Filename not found:", filename)
            return

    os.remove(
        os.path.join(
            folder,
            filename
        )
    )

    update_counts()

    print(
        "Deleted:",
        selected_class,
        filename
    )

delete_image_button.on_click(delete_specific_image)

def predict_live(change):
    global current_prediction

    frame = camera.value

    if frame is None:
        return

    camera_view.value = bgr8_to_jpeg(frame)

    image = Image.fromarray(
        frame[:, :, ::-1]
    )

    x = transform(image)
    x = x.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(x)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        confidence, predicted = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = class_names[
        predicted.item()
    ]

    current_prediction = predicted_class

    confidence_value = (
        confidence.item() * 100
    )

    prediction_label.value = (
        f"Prediction: {predicted_class}"
    )

    confidence_label.value = (
        f"Confidence: {confidence_value:.1f}%"
    )

    check_prediction_change(
        predicted_class
    )

try:
    camera.unobserve(
        _stop_live_callback,
        names="value"
    )
except:
    pass

_stop_live_callback = predict_live

camera.observe(
    _stop_live_callback,
    names="value"
)

display(camera_view)

display(
    prediction_label,
    confidence_label
)

print("DRIVE")

display(
    widgets.HBox([
        start_button,
        stop_button,
        back_button
    ])
)

display(speed_slider)
display(steering)

print("SAVE")

display(
    widgets.HBox([
        save_none_button,
        save_far_button,
        save_close_button
    ])
)

print("RECORD")

display(
    widgets.HBox([
        record_none_button,
        record_far_button,
        record_close_button,
        stop_record_button
    ])
)

display(record_status)

print("COUNTS")

display(
    widgets.HBox([
        none_count,
        far_count,
        close_count
    ])
)

print("DELETE")

display(delete_class)
display(delete_last_button)

display(
    widgets.HBox([
        from_box,
        to_box,
        delete_range_button
    ])
)

display(
    widgets.HBox([
        image_input,
        delete_image_button
    ])
)

print()
print("LIVE STOP DATASET READY")
print("Model: models/best_traffic_stop_model_v2.pth")
print("Prediction change = RECORD STOP + ROBOT STOP")

Device: cuda
NEW STOP V2 model loaded!


Image(value=b'', format='jpeg', height='300', width='300')

Label(value='Prediction: ---')

Label(value='Confidence: ---')

DRIVE


...

COUNTS


DELETE


Dropdown(description='Class:', options=('NONE', 'FAR', 'CLOSE'), value='NONE')

Button(description='DELETE LAST', style=ButtonStyle())


LIVE STOP DATASET READY
Model: models/best_traffic_stop_model_v2.pth
Prediction change = RECORD STOP + ROBOT STOP


In [35]:
camera.stop()
print("Camera stopped")

Camera stopped


In [43]:
from jetbot import Camera

camera = Camera.instance(
    width=224,
    height=224
)

print("Camera ready!")

Camera ready!


In [44]:
import time
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
from jetbot import Robot, bgr8_to_jpeg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# MODELS
# ============================================================

class SteeringCNN(nn.Module):
    def __init__(self):
        super(SteeringCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 5, 2), nn.ReLU(),
            nn.Conv2d(16, 32, 5, 2), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x).squeeze(1)


class StopCNN(nn.Module):
    def __init__(self):
        super(StopCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 5, 2), nn.ReLU(),
            nn.Conv2d(16, 32, 5, 2), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ============================================================
# LOAD MODELS
# ============================================================

steering_model = SteeringCNN().to(device)
steering_model.load_state_dict(
    torch.load("models/clockwise_best_model_3361.pth", map_location=device)
)
steering_model.eval()

stop_model = StopCNN().to(device)
stop_model.load_state_dict(
    torch.load("models/best_traffic_stop_model_v2.pth", map_location=device)
)
stop_model.eval()

print("STEERING model loaded!")
print("NEW STOP V2 model loaded!")

# ============================================================
# TRANSFORM + ROBOT
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

robot = Robot()

# ============================================================
# SETTINGS
# ============================================================

AUTO_SPEED = 0.12

LEFT_MOTOR_GAIN = 1.035
RIGHT_MOTOR_GAIN = 1.00
STEERING_SCALE = 0.10
MAX_STEERING = 0.30

FAR_SPEED = 0.105
STOP_TIME = 5.0

CLOSE_CONFIRM_FRAMES = 2
CLOSE_CONFIDENCE = 0.70

# ============================================================
# STATES
# ============================================================

auto_running = False
stop_until = 0.0
close_count = 0
close_locked = False

# ============================================================
# UI
# ============================================================

camera_view = widgets.Image(format="jpeg", width=320, height=320)

steering_label = widgets.Label(value="Steering: ---")
traffic_label = widgets.Label(value="Traffic: ---")
confidence_label = widgets.Label(value="Confidence: ---")
speed_label = widgets.Label(value="Speed: 0.00")
status_label = widgets.Label(value="Status: READY")

start_button = widgets.Button(
    description="START AUTO",
    button_style="success"
)

stop_button = widgets.Button(
    description="STOP",
    button_style="danger"
)

display(widgets.HBox([start_button, stop_button]))
display(camera_view)
display(steering_label)
display(traffic_label)
display(confidence_label)
display(speed_label)
display(status_label)

# ============================================================
# DRIVE
# ============================================================

def drive(speed, steering):
    steering = max(-MAX_STEERING, min(MAX_STEERING, steering))
    steering_power = steering * STEERING_SCALE

    left_speed = (speed + steering_power) * LEFT_MOTOR_GAIN
    right_speed = (speed - steering_power) * RIGHT_MOTOR_GAIN

    left_speed = max(0.0, min(1.0, left_speed))
    right_speed = max(0.0, min(1.0, right_speed))

    robot.left_motor.value = left_speed
    robot.right_motor.value = right_speed

# ============================================================
# AUTO LOOP
# ============================================================

def auto_drive(change):
    global stop_until, close_count, close_locked

    frame = camera.value
    if frame is None:
        return

    camera_view.value = bgr8_to_jpeg(frame)

    if not auto_running:
        return

    image = Image.fromarray(frame[:, :, ::-1])
    x = transform(image).unsqueeze(0).to(device)

    # Steering
    with torch.no_grad():
        steering_value = steering_model(x).item()

    steering_value = max(
        -MAX_STEERING,
        min(MAX_STEERING, steering_value)
    )

    # Traffic
    with torch.no_grad():
        output = stop_model(x)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, dim=1)

    prediction = predicted.item()
    confidence_value = confidence.item()

    if prediction == 0:
        traffic = "CLOSE"
    elif prediction == 1:
        traffic = "FAR"
    else:
        traffic = "NONE"

    steering_label.value = f"Steering: {steering_value:.3f}"
    traffic_label.value = f"Traffic: {traffic}"
    confidence_label.value = f"Confidence: {confidence_value * 100:.1f}%"

    # 5 second stop
    if time.time() < stop_until:
        robot.stop()
        remaining = stop_until - time.time()
        speed_label.value = "Speed: 0.00"
        status_label.value = f"Status: STOP {remaining:.1f}s"
        return

    # Count CLOSE
    if traffic == "CLOSE" and confidence_value >= CLOSE_CONFIDENCE:
        close_count += 1
    else:
        close_count = 0

    # CLOSE confirmed
    if close_count >= CLOSE_CONFIRM_FRAMES and not close_locked:
        robot.stop()

        stop_until = time.time() + STOP_TIME
        close_locked = True
        close_count = 0

        speed_label.value = "Speed: 0.00"
        status_label.value = "Status: CLOSE -> STOP 5 sec"

        print("CLOSE CONFIRMED -> STOP 5 SEC")
        return

    # Release lock after leaving CLOSE
    if traffic == "NONE":
        close_locked = False

    # Speed
    if traffic == "FAR":
        current_speed = FAR_SPEED
        status_label.value = f"Status: FAR -> SPEED {FAR_SPEED:.2f}"
    else:
        current_speed = AUTO_SPEED
        status_label.value = "Status: DRIVING"

    # Original steering
    drive(current_speed, steering_value)

    speed_label.value = f"Speed: {current_speed:.3f}"

# ============================================================
# START / STOP
# ============================================================

def start_auto(button):
    global auto_running, stop_until, close_count, close_locked

    stop_until = 0.0
    close_count = 0
    close_locked = False
    auto_running = True

    status_label.value = "Status: AUTO RUNNING"
    print("AUTO STARTED")


def stop_auto(button):
    global auto_running, stop_until, close_count, close_locked

    auto_running = False
    stop_until = 0.0
    close_count = 0
    close_locked = False

    robot.stop()

    speed_label.value = "Speed: 0.00"
    status_label.value = "Status: STOPPED"

    print("AUTO STOPPED")


start_button.on_click(start_auto)
stop_button.on_click(stop_auto)

# ============================================================
# CAMERA CALLBACK
# ============================================================

try:
    camera.unobserve(_auto_callback, names="value")
except:
    pass

_auto_callback = auto_drive
camera.observe(_auto_callback, names="value")

print("AUTO READY")
print("STOP model: best_traffic_stop_model_v2.pth")

STEERING model loaded!
NEW STOP V2 model loaded!


Image(value=b'', format='jpeg', height='320', width='320')

Label(value='Steering: ---')

Label(value='Traffic: ---')

Label(value='Confidence: ---')

Label(value='Speed: 0.00')

Label(value='Status: READY')

AUTO READY
STOP model: best_traffic_stop_model_v2.pth


In [45]:
camera.stop()
print("Camera stopped")

Camera stopped
